In [1]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from pymilvus import connections, Collection
import pandas as pd
from underthesea import ner
import re
import unicodedata
import string
import numpy as np

# Kết nối Milvus

In [ ]:
connections.connect("default", host="localhost", port="19530")
collection = Collection("hotels_collection_mpnet_base_v2")

# SBERT - BM25

In [ ]:
def removed_puncts(input_string):
    return input_string.translate(str.maketrans('','',string.punctuation)).lower()

In [ ]:
# Lấy toàn bộ corpus trước, batch để tránh OOM
batch_size = 500
offset = 0
all_docs = []

while True:
    batch = collection.query(
        expr="",  # không filter
        offset=offset,
        limit=batch_size,
        output_fields=["HotelID", "Description"],
    )
    
    if not batch:  # hết data
        break
    
    all_docs.extend(batch)
    offset += batch_size

In [ ]:
all_descriptions = [d["Description"] for d in all_docs]

In [ ]:
def clean_and_tokenize(docs_list):
    tokenized_corpus = []
    for doc in docs_list:
        clean_token_doc = removed_puncts(doc).split()
        tokenized_corpus.append(clean_token_doc)
    return tokenized_corpus

In [ ]:
tokenized_clean_corpus = clean_and_tokenize(all_descriptions)
bm25_corpus_full = BM25Okapi(tokenized_clean_corpus)

In [14]:
model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

In [15]:
def detect_city(query):
    cities = {
        "hồ chí minh": "Hồ Chí Minh",
        "tp hồ chí minh": "Hồ Chí Minh",
        "tp.hcm": "Hồ Chí Minh",
        "sài gòn": "Hồ Chí Minh",
        "hà nội": "Hà Nội",
        "đà nẵng": "Đà Nẵng",
        "phú quốc": "Phú Quốc",
        "nha trang": "Nha Trang",
        "hội an": "Hội An",
        "đà lạt": "Đà Lạt",
        "sa pa": "Sa Pa",
        "sapa": "Sa Pa",
        "huế": "Huế",
        "vũng tàu": "Vũng Tàu"
    }
    query_lower = query.lower()
    
    # Rule-based
    for k, v in cities.items():
        if k in query_lower:
            print(f'Địa danh nhận dạng: {v}')
            return v
    
    # NER-based
    for word, _, _, tag in ner(query):
        if tag.endswith("LOC"):
            # if word.title() in df["Location"].unique():
            expr = f'Location == "{word}"'
            results = collection.query(expr=expr, output_fields=["Location"])
            if results:
                print(f'Địa danh nhận dạng: {word}')
                return word.title()
    
    return None

def clean_text_for_query(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize('NFC', text)
    text = text.lower()

    text = re.sub(r"[^\w\s/\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [16]:
def prepare_query(query):
    city = detect_city(query)
    tokenized_query = removed_puncts(query).split(" ")
    expr = ""
    bm25_scores = []
    filtered_results = []
    
    if city:
        expr = f'Location like "%{city.lower()}%"'
        
        filtered_results = collection.query(
            expr=expr,
            output_fields=["HotelID", "Description"],
        )
        
        if filtered_results:
            filtered_docs = [r['Description'] for r in filtered_results]
            tokenized_filtered = clean_and_tokenize(filtered_docs)
            bm25_local = BM25Okapi(tokenized_filtered)
            bm25_scores = bm25_local.get_scores(tokenized_query)   

    # Nếu không có city hoặc query filter rỗng
    if not filtered_results:
        filtered_results = all_docs
        bm25_scores = bm25_corpus_full.get_scores(tokenized_query)

    # Normalize an toàn
    bm25_scores = np.array(bm25_scores)
    if bm25_scores.max() != bm25_scores.min():
        bm25_scores = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min())
    else:
        bm25_scores = [0] * len(bm25_scores)

    semantic_query = query.lower().replace(city.lower(), "").strip() if city else query.lower()
    
    return semantic_query, expr, bm25_scores, filtered_results

In [ ]:
def hybrid_search(query, alpha=1, top_k=20, debug=True):
    
    semantic_query, expr, bm25_scores, filtered_results = prepare_query(query)
    semantic_query = clean_text_for_query(semantic_query)

    # Tạo dict mapping HotelID -> BM25 score
    bm25_dict = {r['HotelID']: score for r, score in zip(filtered_results, bm25_scores)}
    
    query_emb = model.encode([semantic_query], normalize_embeddings=True)
    search_params = search_params = {"metric_type": "COSINE", "params": {"ef": 64}}

    results = collection.search(
        data=query_emb,
        anns_field="TextForEmbedding",
        param=search_params,
        limit=top_k,
        expr=expr,
        output_fields=["HotelID", "Description", "NameHotel", "Location"]
    )
    
    milvus_hits = []
    
    for hits in results:
        for hit in hits:
            hotel_id = hit.entity.get("HotelID")
            bm25_score = bm25_dict.get(hotel_id, 0)
            milvus_hits.append({
                "HotelID": hit.entity.get("HotelID"),
                "Name Hotel": hit.entity.get("NameHotel"),
                "Descriptions": hit.entity.get("Description"),
                "semantic_score": hit.distance,
                "bm25_score": bm25_score,
                "location" : hit.entity.get("Location"),
            })
            

    for h in milvus_hits:
            h["final_score"] = alpha * h["semantic_score"] + (1 - alpha) * h["bm25_score"]

    # Sắp xếp theo final score
    milvus_hits = sorted(milvus_hits, key=lambda x: x["final_score"], reverse=True)[:top_k]
        
    ids = [str(h['HotelID']) for h in milvus_hits]
    print("HotelID list:",",".join(ids))
    print("=" * 80)

    # In kết quả
    for h in milvus_hits:
        sem = round(h['semantic_score'], 4)
        bm25 = round(h['bm25_score'], 4)
        final = round(h['final_score'], 4)

        print(f"{h['Name Hotel']} (HotelId: {h['HotelID']}) (Score: {final}) (Semantic: {sem}) (BM25: {bm25}) (Location: {h['location']})")
        
        if debug:
            print(f"\n[DEBUG] α={alpha} | Sem={sem:.4f} | BM25={bm25:.4f} → Final={final:.4f}")
            print(f"        Tính tay: {alpha*sem + (1-alpha)*bm25:.4f}")
            
        print(f"\n{h['Descriptions']}\n")

# Test

In [18]:
query = "Resort lãng mạn cho cặp đôi ở Đà Nẵng"
# query = "Resort gần biển yên tĩnh"
# query = "Khách sạn cho gia đình có trẻ em"
# query = "khách sạn gần biển"
hybrid_search(query)

Địa danh nhận dạng: Đà Nẵng
HotelID list: 3333,3038,3937,2756,4397,1393,3141,3178,4409,2325,2044,893,2316,274,937,276,3016,2483,2813,729
ideal villa furama in luxury resort (HotelId: 3333) (Score: 0.7373) (Semantic: 0.7373) (BM25: 0.6716) (Location: đà nẵng)

[DEBUG] α=1 | Sem=0.7373 | BM25=0.6716 → Final=0.7373
        Tính tay: 0.7373

tọa lạc tại thành phố đà nẵng, ideal villa in luxury resort có nhà hàng, lễ tân 24 giờ, quầy bar, vườn, hồ bơi ngoài trời, sân hiên và tầm nhìn ra hồ bơi. chỗ nghỉ cũng có máy điều hòa và bồn tắm spa. mỗi biệt thự tại đây đều có ban công, bếp đầy đủ tiện nghi với lò vi sóng, khu vực ghế ngồi với ghế sofa, tv màn hình phẳng và phòng tắm riêng đi kèm chậu rửa vệ sinh bidet cùng máy sấy tóc. tủ lạnh, bếp nấu và ấm đun nước cũng được trang bị trong phòng. trong khuôn viên chỗ nghỉ có khu vực bãi biển riêng. bãi biển bắc mỹ an nằm cách biệt thự 300 m trong khi bãi biển mỹ khê cách đó 400 m. sân bay gần nhất là sân bay quốc tế đà nẵng, nằm trong bán kính 7 k